# MH AI Chatbot on Google Colab

Runs the chatbot with a GPU-hosted model. Everything is configured in the next cell:
the repository, the databases to fetch and the Hugging Face model to use.

1. `Runtime > Change runtime type` and pick a GPU.
2. Fill in the configuration cell.
3. `Runtime > Run all`, then open the printed link.

The session is temporary. Use the Docker setup in the README for a permanent deployment.

In [ ]:
REPO_URL = "https://github.com/S0NES/MH-AI-chatbot.git"
BRANCH = "main"

# Databases: "<game-id>=<git-url>[#sub-folder]", separated by spaces (see README, "Adding a database")
DATA_SOURCES = ""

# Model: any GGUF chat model on Hugging Face (see README, "Choosing a model")
MHBOT_MODEL_REPO = ""
MHBOT_MODEL_FILE = ""
MHBOT_N_CTX = "4096"
MHBOT_N_GPU_LAYERS = "-1"
MHBOT_CHAT_FORMAT = ""

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU attached to this runtime"
assert MHBOT_MODEL_REPO and MHBOT_MODEL_FILE, "Set MHBOT_MODEL_REPO and MHBOT_MODEL_FILE"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install "llama-cpp-python[server]" huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
!rm -rf /content/app && git clone --depth 1 -b {BRANCH} {REPO_URL} /content/app
!cd /content/app && python scripts/fetch_data.py {DATA_SOURCES}
!cd /content/app && python -m mhbot.check

In [ ]:
import os, subprocess, sys, time, urllib.request

model_env = dict(os.environ, MHBOT_MODELS_DIR="/content/models", MHBOT_MODEL_REPO=MHBOT_MODEL_REPO,
                 MHBOT_MODEL_FILE=MHBOT_MODEL_FILE, MHBOT_N_CTX=MHBOT_N_CTX,
                 MHBOT_N_GPU_LAYERS=MHBOT_N_GPU_LAYERS, MHBOT_CHAT_FORMAT=MHBOT_CHAT_FORMAT)
llm = subprocess.Popen([sys.executable, "-m", "mhbot.model", "serve"], cwd="/content/app", env=model_env,
                       stdout=open("/content/llm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8081/v1/models", timeout=2)
        print("model server ready")
        break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/llm.log").read()[-2000:])
    raise SystemExit("model server did not start")

In [ ]:
web_env = dict(os.environ, HOST="127.0.0.1", PORT="8000", LLM_BASE_URL="http://127.0.0.1:8081/v1")
web = subprocess.Popen([sys.executable, "-m", "mhbot.server"], cwd="/content/app", env=web_env,
                       stdout=open("/content/web.log", "w"), stderr=subprocess.STDOUT)
time.sleep(3)
print(open("/content/web.log").read())

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import re
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if found:
        print("Open:", found.group())
        break